## Exploratory notebook setup

See `docs/notebooks.md` for inputs, execution order, and environment setup.
Run a working copy prepared by `tools/prepare_workspace.py`.
Saved outputs were removed from this copy; executed originals remain on the NAS.
The following cell changes file locations only. The analysis formulas are preserved.


In [ ]:
import os
from pathlib import Path

# Set these before launching Jupyter when using another storage mount.
VQ_NAS_ROOT = Path(os.environ.get("VQ_NAS_ROOT", "/megaverse/storage/nasiri"))
VQ_JUPYTER_ROOT = Path(os.environ.get("VQ_JUPYTER_ROOT", str(VQ_NAS_ROOT / "jupyter")))


# Encoder Locality Analysis

## Goal (H1: patch pixel-noise on encoder)
We perturb the **input image** with Gaussian noise **only inside a pixel-space patch** and re-encode to discrete token IDs.
The question is: **how often do token IDs change outside the token-aligned patch region?**

Each sample provides:
- `token_grid_hw = (H, W)` token grid resolution
- `patch_bbox_px = (x0, y0, x1, y1)` pixel-space patch (where noise was applied)
- `patch_bbox_tok = (j0, i0, j1, i1)` token-space bounding box aligned to the token grid
- `indices_clean`, `indices_low`, `indices_mid`, `indices_high`: flattened token IDs (length `H*W`)


## Setup

In [ ]:
from pathlib import Path
import glob
import json
import pandas as pd
import numpy as np
import random
from PIL import Image
import matplotlib.pyplot as plt

OUTDIR = VQ_JUPYTER_ROOT / "1766315949_robustness_dataset_vqgan_h1_patch_noise_encoder"

meta_files = sorted(OUTDIR.glob("metadata_part_*.jsonl"))
print("OUTDIR:", OUTDIR)
print("Found metadata files:", len(meta_files))
for p in meta_files[:10]:
    print(" -", p.name)
assert meta_files, "No metadata_part_*.jsonl files found. Fix OUTDIR."

# ---- codebook ----
# Expect shape [K, D]
codebook_path = OUTDIR / "codebook.npy"
assert codebook_path.exists(), f"Missing codebook.npy at {codebook_path}"
codebook = np.load(codebook_path)  # float32/float16
assert codebook.ndim == 2, f"Expected codebook to be 2D [K, D], got shape={codebook.shape}"
K, D = codebook.shape
print(f"Loaded codebook: K={K}, D={D}, dtype={codebook.dtype}")

# Pre-normalize for cosine similarity (dot product)
codebook = codebook.astype(np.float32, copy=False)
norms = np.linalg.norm(codebook, axis=1, keepdims=True)
codebook_unit = codebook / np.clip(norms, 1e-12, None)

In [ ]:
MAX_RECORDS = None  # set e.g. 2000 for a quick first pass

records = []
for fp in meta_files:
    with open(fp, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
            if MAX_RECORDS is not None and len(records) >= MAX_RECORDS:
                break
    if MAX_RECORDS is not None and len(records) >= MAX_RECORDS:
        break

df = pd.DataFrame(records)
print("Loaded records:", len(df))
print("Columns:", list(df.columns))

possible_tok_bbox_cols = [c for c in df.columns if "bbox" in c and ("tok" in c or "token" in c)]
possible_px_bbox_cols  = [c for c in df.columns if "bbox" in c and ("px" in c or "pixel" in c)]
print("Possible token-bbox columns:", possible_tok_bbox_cols)
print("Possible pixel-bbox columns:", possible_px_bbox_cols)

df.head(3)

In [ ]:
IMAGES_DIR = OUTDIR / "images"

img_files = sorted([p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in {".png"}])
print("IMAGES_DIR:", IMAGES_DIR)
print("Total image files found:", len(img_files))
print("Sample filenames:")
for p in random.sample(img_files, k=min(15, len(img_files))):
    print(" -", p.relative_to(IMAGES_DIR))

### Sanity Check

In [ ]:
required = ["indices_clean", "indices_low", "indices_mid", "indices_high",
            "patch_bbox_tok", "token_grid_hw", "noise_std"]
missing = [k for k in required if k not in df.columns]
assert not missing, f"Missing required fields: {missing}"

# Normalise noise_std to a tuple (hashable) and also keep explicit named stds
df["noise_std_tuple"] = df["noise_std"].apply(tuple)

# Most runs have a single triple like (low, mid, high); we assert that here
uniq_noise = df["noise_std_tuple"].unique()
print("Unique noise_std tuples:", uniq_noise[:10])
assert len(uniq_noise) == 1, f"Expected one noise_std triple; got {len(uniq_noise)}"

sigma_low, sigma_mid, sigma_high = uniq_noise[0]
print("σ_low, σ_mid, σ_high =", sigma_low, sigma_mid, sigma_high)

print("token_grid_hw unique (first 10):", df["token_grid_hw"].astype(str).unique()[:10])

# Check a couple rows manually
for i in range(min(3, len(df))):
    H, W = df.loc[i, "token_grid_hw"]
    bbox = df.loc[i, "patch_bbox_tok"]
    print(f"row {i}: grid=({H},{W})  bbox={bbox}  len(clean)={len(df.loc[i,'indices_clean'])}")


In [ ]:
def _bbox_area(b):
    j0, i0, j1, i1 = b
    return max(0, j1 - j0) * max(0, i1 - i0)

H = df["token_grid_hw"].apply(lambda x: x[0]).to_numpy()
W = df["token_grid_hw"].apply(lambda x: x[1]).to_numpy()
HW = H * W

len_clean = df["indices_clean"].apply(len).to_numpy()
len_low   = df["indices_low"].apply(len).to_numpy()
len_mid   = df["indices_mid"].apply(len).to_numpy()
len_high  = df["indices_high"].apply(len).to_numpy()

assert np.all(len_clean == HW), "Some indices_clean lengths != H*W"
assert np.all(len_low == HW) and np.all(len_mid == HW) and np.all(len_high == HW), "Some noisy indices lengths != H*W"

bboxes = df["patch_bbox_tok"].to_list()
j0 = np.array([b[0] for b in bboxes])
i0 = np.array([b[1] for b in bboxes])
j1 = np.array([b[2] for b in bboxes])
i1 = np.array([b[3] for b in bboxes])

assert np.all((0 <= j0) & (j0 <= j1) & (j1 <= W)), "BBox x-bounds invalid for some rows"
assert np.all((0 <= i0) & (i0 <= i1) & (i1 <= H)), "BBox y-bounds invalid for some rows"

patch_area = np.array([_bbox_area(b) for b in bboxes])
patch_frac = patch_area / HW

print("Patch fraction token-space summary:")
print(pd.Series(patch_frac).describe(percentiles=[0.1,0.5,0.9,0.99]))

## Visual Sanity Check

In [ ]:
def find_variants(image_id: str):
    """
    Try to find 1-4 images in OUTDIR/images that belong to this image_id.
    Returns dict with keys among: clean, low, mid, high, any
    """
    # Broad search: any file containing the image_id
    hits = [p for p in img_files if image_id in str(p)]
    out = {}

    if not hits:
        return out

    # Heuristic bucketing by filename
    for p in hits:
        s = p.name.lower()
        if "clean" in s:
            out["clean"] = p
        elif "low" in s:
            out["low"] = p
        elif "mid" in s:
            out["mid"] = p
        elif "high" in s:
            out["high"] = p

    # Fallback: if none matched buckets, take first as "any"
    if not out:
        out["any"] = hits[0]

    return out

def draw_boxes(ax, img_w, img_h, bbox_px, bbox_tok, grid_hw):
    # bbox_px: [x0, y0, x1, y1] in pixels
    x0, y0, x1, y1 = bbox_px
    ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0,
                              fill=False, linewidth=2))

def show_variant(ax, path, title, row):
    img = Image.open(path).convert("RGB")
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    draw_boxes(ax, img.width, img.height,
               row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])


In [ ]:
def find_full_set(image_id: str):
    # Return dict idx(int)->Path for files "0_*.png" ... "7_*.png" under that image_id folder
    # We rely on the exporter’s numbering convention.
    folder_hits = [p for p in img_files if (image_id in str(p))]
    if not folder_hits:
        return {}

    # usually all 8 sit in one folder; infer folder from any hit
    parent = folder_hits[0].parent
    out = {}
    for k in range(8):
        matches = sorted(parent.glob(f"{k}_*.png"))
        if matches:
            out[k] = matches[0]
    return out


In [ ]:
def show_sample_grid(row, overlay=True):
    image_id = row["image_id"]
    files = find_full_set(image_id)

    print("\nimage_id:", image_id)
    print("found:", {k: files[k].name for k in sorted(files.keys())})

    needed = list(range(8))
    if not all(k in files for k in needed):
        print("WARNING: not all 0..7 found for this sample.")
        return

    titles_top = ["original", "input low", "input mid", "input high"]
    titles_bot = ["recon clean", "recon low", "recon mid", "recon high"]

    top_paths = [files[0], files[2], files[4], files[6]]
    bot_paths = [files[1], files[3], files[5], files[7]]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    for c, p in enumerate(top_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[0, c]
        ax.imshow(img)
        ax.set_title(titles_top[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_boxes(ax, img.width, img.height,
                       row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])

    for c, p in enumerate(bot_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[1, c]
        ax.imshow(img)
        ax.set_title(titles_bot[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_boxes(ax, img.width, img.height,
                       row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])

    plt.tight_layout()
    plt.show()

N = 3
for r in df.sample(N, random_state=2).to_dict(orient="records"):
    show_sample_grid(r, overlay=True)


## Definitions and Metrics

Let the clean token grid be  
$ \mathbf{z} \in \{0,\dots,K-1\}^{H \times W} $  
and the perturbed token grid at noise level $\sigma$ be  
$ \mathbf{z}_{\sigma} $.

Let the VQ codebook be $E \in \mathbb{R}^{K \times D}$, where $E[k]$ is the embedding of token id $k$.
Define the $\ell_2$-normalised embeddings  
$ \hat{E}[k] = \dfrac{E[k]}{\|E[k]\|_2 + \delta} $ with small $\delta>0$.

### Per-token embedding distance (scaled to [0,1])
For each position $(i,j)$, define cosine similarity:
$$
s_{\sigma}(i,j)
=
\left\langle \hat{E}\!\left[z(i,j)\right],\; \hat{E}\!\left[z_{\sigma}(i,j)\right] \right\rangle
\in [-1,1].
$$

Convert to a bounded distance in $[0,1]$ (from [-1,1] sapce):
$$
\Delta_{\sigma}(i,j)
=
\frac{1 - s_{\sigma}(i,j)}{2}
\in [0,1].
$$

Notes: $\Delta_{\sigma}(i,j)=0$ when the token id is unchanged (same embedding), and approaches $1$ when the two embeddings are maximally dissimilar under cosine.

### Patch region
Let the token patch box be  
$ B = (j_0,i_0,j_1,i_1) $  
and define the set of token positions inside it:
$$
\mathcal{P}
= \{(i,j)\mid i_0 \le i < i_1,\; j_0 \le j < j_1\},
\qquad
\mathcal{P}^c \text{ its complement.}
$$

### Inside / Outside change (mean embedding distance)
**Inside-change**
$$
\text{inside\_change}(\sigma)
= \frac{1}{|\mathcal{P}|}
\sum_{(i,j)\in\mathcal{P}} \Delta_{\sigma}(i,j)
$$

**Outsid**


In [ ]:
# Grid is constant (16x16)
H = df["token_grid_hw"].apply(lambda x: x[0]).to_numpy()
W = df["token_grid_hw"].apply(lambda x: x[1]).to_numpy()
HW = H * W

assert np.all(HW == HW[0]), "token_grid_hw not constant across rows"
H0, W0, HW0 = int(H[0]), int(W[0]), int(HW[0])

# Build inside mask per sample: shape (N, HW)
N = len(df)
inside_mask = np.zeros((N, HW0), dtype=bool)

bboxes = df["patch_bbox_tok"].to_list()
for n, (j0, i0, j1, i1) in enumerate(bboxes):
    m2d = inside_mask[n].reshape(H0, W0)   # row=i, col=j
    m2d[i0:i1, j0:j1] = True

inside_count = inside_mask.sum(axis=1).astype(np.float32)
outside_count = (HW0 - inside_count).astype(np.float32)

# Optional: confirm patch area equals bbox area
bbox_area = np.array([(j1-j0)*(i1-i0) for (j0,i0,j1,i1) in bboxes], dtype=np.int32)
assert np.all(inside_count == bbox_area), "inside_mask area != bbox area for some rows"

print("Token grid:", (H0, W0), "HW=", HW0)
print("Inside token count summary:", np.percentile(inside_count, [0, 10, 50, 90, 99, 100]))
print("Outside token count summary:", np.percentile(outside_count, [0, 10, 50, 90, 99, 100]))


In [ ]:
import torch
from tqdm.auto import tqdm

# Cache token ids once on CPU (int32)
z_clean_np = np.asarray(df["indices_clean"].to_list(), dtype=np.int32)  # (N, HW)

# Move codebook_unit to GPU (float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
codebook_unit_t = torch.from_numpy(codebook_unit).to(device=device, dtype=torch.float32)

print("Device:", device, "| codebook_unit_t:", tuple(codebook_unit_t.shape), codebook_unit_t.dtype)

def compute_change_metrics_distance_gpu(noisy_col: str, eps: float = 1e-8, chunk_rows: int = 1024):
    """
    Computes inside/outside mean cosine distance in [0,1] for df[noisy_col] vs clean.
    Chunked over rows to control GPU memory.
    Returns numpy float32 arrays of shape (N,).
    """
    z_noisy_np = np.asarray(df[noisy_col].to_list(), dtype=np.int32)  # (N, HW)

    N, HW = z_clean_np.shape
    inside_change = np.empty((N,), dtype=np.float32)
    outside_change = np.empty((N,), dtype=np.float32)
    leakage_ratio  = np.empty((N,), dtype=np.float32)

    # Move masks/counts once (they're small)
    inside_mask_t  = torch.from_numpy(inside_mask.astype(np.float32)).to(device)   # (N, HW) float mask
    outside_mask_t = 1.0 - inside_mask_t
    inside_count_t = torch.from_numpy(inside_count).to(device)                      # (N,)
    outside_count_t= torch.from_numpy(outside_count).to(device)                     # (N,)

    for s in tqdm(range(0, N, chunk_rows), desc=f"cos-dist metrics: {noisy_col}"):
        e = min(N, s + chunk_rows)

        zc = torch.from_numpy(z_clean_np[s:e]).to(device=device, dtype=torch.long)  # (B, HW)
        zn = torch.from_numpy(z_noisy_np[s:e]).to(device=device, dtype=torch.long)  # (B, HW)

        # Gather embeddings: (B, HW, D)
        Ec = codebook_unit_t[zc]
        En = codebook_unit_t[zn]

        # Cos sim = dot since unit-normalized: (B, HW)
        cos_sim = (Ec * En).sum(dim=-1).clamp(-1.0, 1.0)
        delta   = (1.0 - cos_sim) * 0.5  # [0,1], (B, HW)

        im = inside_mask_t[s:e]
        om = outside_mask_t[s:e]

        inside_mean  = (delta * im).sum(dim=1) / inside_count_t[s:e]
        outside_mean = (delta * om).sum(dim=1) / outside_count_t[s:e]
        ratio        = outside_mean / (inside_mean + eps)

        inside_change[s:e] = inside_mean.detach().float().cpu().numpy()
        outside_change[s:e]= outside_mean.detach().float().cpu().numpy()
        leakage_ratio[s:e] = ratio.detach().float().cpu().numpy()

    return inside_change, outside_change, leakage_ratio

inside_low, outside_low, ratio_low     = compute_change_metrics_distance_gpu("indices_low",  chunk_rows=2048)
inside_mid, outside_mid, ratio_mid     = compute_change_metrics_distance_gpu("indices_mid",  chunk_rows=2048)
inside_high, outside_high, ratio_high  = compute_change_metrics_distance_gpu("indices_high", chunk_rows=2048)

print("Computed GPU distance-based metrics for low/mid/high.")

In [ ]:
def summarize(arr):
    return {
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "q90": float(np.quantile(arr, 0.90)),
        "q99": float(np.quantile(arr, 0.99)),
    }

summary = pd.DataFrame({
    "sigma": [sigma_low, sigma_mid, sigma_high],
    "inside_change": [summarize(inside_low), summarize(inside_mid), summarize(inside_high)],
    "outside_change": [summarize(outside_low), summarize(outside_mid), summarize(outside_high)],
    "leakage_ratio": [summarize(ratio_low), summarize(ratio_mid), summarize(ratio_high)],
})

def expand(col):
    return pd.DataFrame(summary[col].tolist()).add_prefix(f"{col}_")

out = pd.concat(
    [summary[["sigma"]], expand("inside_change"), expand("outside_change"), expand("leakage_ratio")],
    axis=1
)
out


**Results: Locality vs Leakage**

We quantify encoder locality by measuring the **mean cosine embedding distance** between the clean token and the noisy token at each position:
$\Delta_\sigma(i,j) \in [0,1]$, where $0$ means identical (same token embedding) and larger values mean a bigger semantic move in codebook space.

**Inside response**  
Inside the token-aligned patch, the encoder response grows strongly with noise:
- At $\sigma=0.1$, median inside-change $\approx 0.078$ (mean $\approx 0.089$).
- At $\sigma=0.5$, median inside-change $\approx 0.288$ (mean $\approx 0.291$).

So patch-local noise produces increasingly large embedding shifts in the patch tokens, but it does **not** “saturate to 1” because we are no longer counting flips; we’re measuring *how far* tokens move in embedding space.

**Outside response (leakage / non-locality)**  
Outside the patch, the change is smaller but clearly non-zero and increases with noise:
- At $\sigma=0.1$, median outside-change $\approx 0.0163$ (mean $\approx 0.0184$).
- At $\sigma=0.5$, median outside-change $\approx 0.0424$ (mean $\approx 0.0456$).

This means a perturbation confined to a local pixel patch still induces a measurable embedding drift across the rest of the image tokens.

**Normalised leakage ratio**  
Using the ratio of outside to inside mean distance:
- $\sigma=0.1$: median leakage ratio $\approx 0.211$
- $\sigma=0.2$: median leakage ratio $\approx 0.182$
- $\sigma=0.5$: median leakage ratio $\approx 0.150$

So **relative leakage decreases with noise**: the patch region responds more strongly than the outside as $\sigma$ grows, even though absolute outside change still increases.

**Tail behaviour**  
The leakage ratio has a heavy tail: the 99th percentile remains high (≈0.61 at $\sigma=0.1$, ≈0.46 at $\sigma=0.2$, ≈0.33 at $\sigma=0.5$), indicating a subset of images where patch-local noise produces unusually strong non-local effects.


## Spatial Leakage Profile: mean embedding change vs distance from patch

Scalar outside-change cannot distinguish *boundary spillover* from *long-range propagation*.
We therefore compute a distance-to-patch curve.

For each token position $(i,j)$ on the $H \times W$ grid, define its Manhattan distance to the patch:
- $d(i,j)=0$ if $(i,j)\in\mathcal{P}$
- otherwise $d(i,j)$ is the Manhattan distance to the nearest patch boundary.

For each noise level $\sigma$, we estimate the **mean per-token embedding distance** at each distance:
$$
p_{\sigma}(d)
=
\mathbb{E}\left[
\frac{1}{|\{(i,j): d(i,j)=d\}|}
\sum_{(i,j): d(i,j)=d}
\Delta_{\sigma}(i,j)
\right],
\qquad d=0,1,2,\dots
$$

If leakage is mostly local, $p_{\sigma}(d)$ should drop sharply after $d=0$ (and ideally after $d=1$).
A slow decay indicates long-range non-locality.


In [ ]:
# Precompute a base (i,j) grid once (since H,W are constant here)
ii, jj = np.meshgrid(np.arange(H0), np.arange(W0), indexing="ij")  # (H,W)

bboxes = df["patch_bbox_tok"].to_list()
dist_map = np.zeros((N, HW0), dtype=np.int16)

for n, (j0, i0, j1, i1) in enumerate(bboxes):
    # distance to interval [i0, i1-1] in rows, [j0, j1-1] in cols
    # (0 if inside)
    di = np.where(ii < i0, i0 - ii, np.where(ii > (i1 - 1), ii - (i1 - 1), 0))
    dj = np.where(jj < j0, j0 - jj, np.where(jj > (j1 - 1), jj - (j1 - 1), 0))
    d = (di + dj).astype(np.int16)  # Manhattan distance to the box
    dist_map[n] = d.reshape(-1)

max_d = int(dist_map.max())
print("Max Manhattan distance observed:", max_d)
print("Distances present:", np.unique(dist_map)[:20], "...", np.unique(dist_map)[-10:])


In [ ]:
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# dist_map is (N, HW) int16; move once to GPU for fast masking
dist_map_t = torch.from_numpy(dist_map.astype(np.int16)).to(device)
max_d = int(dist_map.max())

def p_of_d_distance_gpu(noisy_col: str, chunk_rows: int = 2048):
    z_noisy_np = np.asarray(df[noisy_col].to_list(), dtype=np.int32)  # (N, HW)

    sum_by_d = torch.zeros((max_d + 1,), device=device, dtype=torch.float64)
    cnt_by_d = torch.zeros((max_d + 1,), device=device, dtype=torch.float64)

    for s in tqdm(range(0, N, chunk_rows), desc=f"p(d) cos-dist: {noisy_col}"):
        e = min(N, s + chunk_rows)
        B = e - s

        zc = torch.from_numpy(z_clean_np[s:e]).to(device=device, dtype=torch.long)  # (B, HW)
        zn = torch.from_numpy(z_noisy_np[s:e]).to(device=device, dtype=torch.long)  # (B, HW)

        Ec = codebook_unit_t[zc]  # (B, HW, D)
        En = codebook_unit_t[zn]
        cos_sim = (Ec * En).sum(dim=-1).clamp(-1.0, 1.0)  # (B, HW)
        delta = (1.0 - cos_sim) * 0.5                     # (B, HW)

        dm = dist_map_t[s:e]  # (B, HW)

        for d in range(max_d + 1):
            m = (dm == d)  # (B, HW) bool
            cnt = m.sum().item()
            if cnt == 0:
                continue
            sum_by_d[d] += (delta[m]).sum(dtype=torch.float64)
            cnt_by_d[d] += float(cnt)

    p = (sum_by_d / torch.clamp(cnt_by_d, min=1.0)).float().cpu().numpy()
    cnt = cnt_by_d.long().cpu().numpy()
    return p.astype(np.float32), cnt.astype(np.int64)


p_low, cnt_d  = p_of_d_distance_gpu("indices_low",  chunk_rows=2048)
p_mid, _      = p_of_d_distance_gpu("indices_mid",  chunk_rows=2048)
p_high, _     = p_of_d_distance_gpu("indices_high", chunk_rows=2048)

pd.DataFrame({
    "d": np.arange(max_d + 1),
    "count_tokens_total": cnt_d,
    f"mean_dist(d)|σ={sigma_low}": p_low,
    f"mean_dist(d)|σ={sigma_mid}": p_mid,
    f"mean_dist(d)|σ={sigma_high}": p_high,
})

**Table: Distance-to-patch response (embedding-distance).**  
For each Manhattan distance \(d\) from the edited patch, this table reports $$p_\sigma(d)$$, the mean per-token cosine embedding distance $$\Delta_\sigma(i,j)\in[0,1]$$ between clean and noisy token embeddings, averaged over all tokens with \(d(i,j)=d\) and over all images. Here \(d=0\) corresponds to tokens inside the patch. `count_tokens_total` is the total number of token positions contributing at each distance across the dataset.


In [ ]:
d = np.arange(max_d + 1)

plt.figure(figsize=(7, 4))
plt.plot(d, p_low,  label=f"σ={sigma_low}")
plt.plot(d, p_mid,  label=f"σ={sigma_mid}")
plt.plot(d, p_high, label=f"σ={sigma_high}")
plt.xlabel("Manhattan distance to patch (tokens)")
plt.ylabel("Mean cosine embedding distance")
plt.title("Spatial leakage profile (embedding-distance)")
plt.legend()
plt.ylim(0, 0.35)
plt.yticks(np.arange(0, 0.351, 0.05))
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
eps = 1e-8
p_low  = np.clip(p_low,  eps, None)
p_mid  = np.clip(p_mid,  eps, None)
p_high = np.clip(p_high, eps, None)

d = np.arange(max_d + 1)

plt.figure(figsize=(7, 4))
plt.plot(d, p_low,  label=f"σ={sigma_low}")
plt.plot(d, p_mid,  label=f"σ={sigma_mid}")
plt.plot(d, p_high, label=f"σ={sigma_high}")
plt.xlabel("Manhattan distance to patch (tokens)")
plt.ylabel("Mean cosine embedding distance(log)")
plt.title("Spatial leakage profile (embedding-distance)")
plt.legend()

plt.yscale("log")
plt.grid(True, which="both", alpha=0.25)

plt.tight_layout()
plt.show()

**Plot: Spatial leakage profile (embedding-distance).**  
This curve visualises $$p_\sigma(d)=\mathbb{E}[\Delta_\sigma(i,j)\mid d(i,j)=d]$$, where $$\Delta_\sigma(i,j)=\tfrac{1-\langle \hat E[z(i,j)],\hat E[z_\sigma(i,j)]\rangle}{2}$$ is the cosine embedding distance in \([0,1]\). The point \(d=0\) is the patch region; larger \(d\) moves away from the patch. A sharp drop after \(d=0\) indicates mostly local effects, while a non-zero tail at large \(d\) indicates long-range drift (non-local leakage).


In [ ]:
def curves_flip_and_dist(noisy_col: str):
    z_noisy = np.asarray(df[noisy_col].to_list(), dtype=np.int32)

    flip = (z_noisy != z_clean_np)  # (N, HW) bool

    p_flip = []
    for d in range(max_d + 1):
        m = (dist_map == d)
        p_flip.append(float(flip[m].mean()) if m.any() else np.nan)
    p_flip = np.array(p_flip, dtype=np.float32)

    return p_flip

pflip_low  = curves_flip_and_dist("indices_low")
pflip_mid  = curves_flip_and_dist("indices_mid")
pflip_high = curves_flip_and_dist("indices_high")


In [ ]:
d = np.arange(max_d + 1)
plt.figure(figsize=(7,4))
plt.plot(d, pflip_low,  label=f"σ={sigma_low}")
plt.plot(d, pflip_mid,  label=f"σ={sigma_mid}")
plt.plot(d, pflip_high, label=f"σ={sigma_high}")
plt.xlabel("Manhattan distance to patch (tokens)")
plt.ylabel("P(token ID changes)")
plt.title("Spatial leakage profile (flip probability)")
plt.ylim(0, 1)
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

**Plot: Spatial leakage profile (flip probability).**  
This curve shows the probability that a token *ID* changes as a function of distance to the patch:
$$
p^{\text{flip}}_{\sigma}(d)=\Pr\!\big[z_\sigma(i,j)\neq z(i,j)\,\big|\, d(i,j)=d\big].
$$
Here \(d=0\) corresponds to tokens inside the patch. Compared to embedding-distance, this plot measures **how often** tokens change (discrete flips), not **how far** they move in codebook space. A slow decay with \(d\) indicates that patch-local noise causes widespread token reassignment beyond the patch.


**Results: Spatial leakage profile**

We compute a distance-conditioned response curve $p_{\sigma}(d)$, where $d$ is Manhattan distance (in token units) to the token-aligned patch box, and $p_{\sigma}(d)$ is the **mean cosine embedding distance** $\Delta_{\sigma}(i,j)\in[0,1]$ at positions with $d(i,j)=d$.

Across all noise levels, $p_{\sigma}(d)$ drops sharply from $d=0$ to small distances ($d\approx 1\text{–}3$), confirming that the patch region is affected most strongly, with rapidly decaying spillover near the boundary.

However, $p_{\sigma}(d)$ does **not** decay to zero at large distances. Instead it approaches a small but non-zero plateau (global drift):
- for $\sigma=0.1$, $p_{\sigma}(d)$ stabilizes around $\approx 0.007\text{–}0.009$,
- for $\sigma=0.2$, the plateau is around $\approx 0.011\text{–}0.013$,
- for $\sigma=0.5$, the plateau is around $\approx 0.016\text{–}0.019$.

This indicates a measurable **non-local effect**: patch-local pixel noise induces a distributed embedding shift in token assignments even far from the patch, and the magnitude of this far-field drift increases with $\sigma$.

Finally, the largest distances involve very few token positions (small counts in the table), so tail fluctuations should be interpreted cautiously.


## Qualitative Visual Examples (Embedding-distance heatmaps)

This visualization shows **how far token representations move in codebook space** after injecting patch-local noise, compared to the clean encoding.

Each cell corresponds to one token position in the \(H \times W\) latent grid and is coloured by the **cosine embedding distance**
$$
\Delta_{\sigma}(i,j)=\frac{1-\langle \hat E[z(i,j)],\hat E[z_{\sigma}(i,j)]\rangle}{2}\in[0,1].
$$

* **Dark / near 0**: little to no change (same or very similar token embedding).
* **Bright / near 1**: large change (token embedding becomes very dissimilar).

The black rectangle indicates the **edited patch region** in token space. High values inside the box reflect the **direct effect** of the injected noise, while elevated values outside the box indicate **non-local effects (spatial leakage)**.

In [ ]:
# Choose which leakage metric to rank on (distance-based)
rank_ratio = ratio_mid  # ratio_low / ratio_high also ok

idx_sorted = np.argsort(rank_ratio)
low_idxs  = idx_sorted[:3]
high_idxs = idx_sorted[-3:][::-1]

# Median-ish sample (closest to median leakage ratio)
med_val = np.median(rank_ratio)
med_idx = int(np.argmin(np.abs(rank_ratio - med_val)))

# Random "ordinary" samples from the middle band (avoid extremes)
rng = np.random.default_rng(0)
lo_q, hi_q = np.quantile(rank_ratio, [0.2, 0.8])
mid_pool = np.where((rank_ratio >= lo_q) & (rank_ratio <= hi_q))[0]
rand_idxs = rng.choice(mid_pool, size=3, replace=False).astype(int)

def print_row(i, tag=""):
    print(f"{tag} idx={i}  image_id={df.loc[i,'image_id']}  "
          f"ratio_mid={float(rank_ratio[i]):.4f}  "
          f"outside_mid={float(outside_mid[i]):.4f}  inside_mid={float(inside_mid[i]):.4f}")

print("Lowest leakage_ratio (mid):")
for i in low_idxs:  print_row(int(i), "LOW ")

print("\nMedian leakage sample:")
print_row(med_idx, "MED ")

print("\nRandom ordinary samples (20–80% band):")
for i in rand_idxs: print_row(int(i), "RND ")

print("\nHighest leakage_ratio (mid):")
for i in high_idxs: print_row(int(i), "HIGH")

In [ ]:
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def token_dist_grid(row, level: str):
    """Return HxW grid of cosine embedding distance in [0,1] for one row."""
    H, W = row["token_grid_hw"]
    zc = np.array(row["indices_clean"], dtype=np.int32).reshape(H, W)
    zn = np.array(row[f"indices_{level}"], dtype=np.int32).reshape(H, W)

    zc_t = torch.from_numpy(zc).to(device=device, dtype=torch.long)
    zn_t = torch.from_numpy(zn).to(device=device, dtype=torch.long)

    Ec = codebook_unit_t[zc_t]  # (H,W,D)
    En = codebook_unit_t[zn_t]
    cos_sim = (Ec * En).sum(dim=-1).clamp(-1.0, 1.0)
    delta = (1.0 - cos_sim) * 0.5
    return delta.detach().cpu().numpy()

In [ ]:
def show_token_heatmaps_3(row_idx: int):
    row = df.loc[row_idx]
    H, W = row["token_grid_hw"]
    j0, i0, j1, i1 = row["patch_bbox_tok"]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)

    ims = []
    for ax, lvl in zip(axes, ["low", "mid", "high"]):
        delta = token_dist_grid(row, lvl)
        im = ax.imshow(delta, vmin=0, vmax=1)
        ims.append(im)

        ax.add_patch(plt.Rectangle((j0-0.5, i0-0.5), (j1-j0), (i1-i0),
                                   fill=False, linewidth=2))
        ax.set_title(f"Embedding-distance ({lvl})")
        ax.axis("off")

    # one shared colorbar for all axes
    fig.colorbar(ims[0], ax=axes, fraction=0.03, pad=0.02, label="cosine distance (0..1)")

    plt.show()


print("LOW leakage sample heatmaps:")
show_token_heatmaps_3(int(low_idxs[0]))

print("MEDIAN leakage sample heatmaps:")
show_token_heatmaps_3(med_idx)

print("RANDOM leakage sample heatmaps:")
show_token_heatmaps_3(int(rand_idxs[0]))

print("HIGH leakage sample heatmaps:")
show_token_heatmaps_3(int(high_idxs[0]))

**Qualitative interpretation (embedding-distance heatmaps)**

Each heatmap cell is a token position on the $16\times16$ grid, coloured by the cosine embedding distance $\Delta\in[0,1]$ between the clean token and the noisy token. The black box marks the perturbed patch ($d=0$).

- **Low leakage sample:** changes are concentrated inside the box, with the outside region mostly dark. This is close to *local* behaviour: the effect of patch noise stays in the patch.
- **Median / random samples:** the patch changes strongly, and a few outside positions brighten as noise increases. This indicates *moderate leakage*: some non-local drift appears beyond the patch, but it remains limited.
- **High leakage sample:** substantial bright regions appear **outside** the box (sometimes far from it) and intensify with noise. This is the key failure mode: patch-local pixel noise induces large representation changes at distant tokens.

**Conjecture (selective non-locality):** the outside changes are not a uniform “global brightening”. Instead, only *specific* distant locations light up while other outside tokens remain dark. This suggests the non-local effect is **selective** (image- and content-dependent), rather than simple boundary spillover. In other words, patch noise can trigger distant token changes in particular salient regions (e.g., strong edges/structures) instead of spreading evenly across the grid.

In [ ]:
def show_random_token_heatmaps_grid(N=10, lvls=("low", "mid", "high"), seed=42):
    rand_idxs = df.sample(n=N, random_state=seed).index

    nrows = N
    ncols = len(lvls)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4 * ncols, 4 * nrows),
        constrained_layout=True
    )

    # handle N=1 edge case
    if nrows == 1:
        axes = axes[None, :]

    for r, idx in enumerate(rand_idxs):
        row = df.loc[idx]
        j0, i0, j1, i1 = row["patch_bbox_tok"]

        for c, lvl in enumerate(lvls):
            ax = axes[r, c]
            delta = token_dist_grid(row, lvl)

            im = ax.imshow(delta, vmin=0, vmax=1)
            ax.add_patch(
                plt.Rectangle(
                    (j0 - 0.5, i0 - 0.5),
                    (j1 - j0),
                    (i1 - i0),
                    fill=False,
                    linewidth=2
                )
            )

            if r == 0:
                ax.set_title(lvl)
            if c == 0:
                ax.set_ylabel(f"id {idx}")

            ax.axis("off")

    fig.colorbar(
        im,
        ax=axes,
        fraction=0.02,
        pad=0.01,
        label="cosine distance (0..1)"
    )

    plt.show()

show_random_token_heatmaps_grid(N=10)

## Evidence check: is outside leakage *selective* (hotspots) or *diffuse* (global drift)?

### Motivation
From qualitative heatmaps, some high-leakage examples appear to show “hotspots” outside the patch. To test whether non-local effects are genuinely **selective** (a small set of outside tokens carries most of the change) versus **diffuse** (outside change spreads broadly), we quantify *concentration* of the outside embedding-distance signal using only the existing exported token grids and codebook.

### Setup
For each image at noise level $\sigma=\sigma_{\text{mid}}$, we compute the per-token cosine embedding distance
$$
\Delta(i,j)=\frac{1-\langle \hat E[z(i,j)], \hat E[z_{\sigma}(i,j)]\rangle}{2}\in[0,1],
$$
and restrict to tokens **outside** the token-aligned patch region $\mathcal{P}^c$.

We keep the standard scalar leakage metrics:
- $\text{outside\_mid}=\mathbb{E}[\Delta(i,j)\mid (i,j)\in\mathcal{P}^c]$
- $\text{ratio\_mid}=\text{outside\_mid}/(\text{inside\_mid}+\varepsilon)$

### Selectivity metrics (outside only)
Let $x$ be the vector of outside-token distances $x=\{\Delta(i,j):(i,j)\in\mathcal{P}^c\}$.
We measure how concentrated the outside signal is via:

- **Top-5% mass** (energy concentration):
$$
\text{top5\_mass}=\frac{\sum_{k\in\text{Top 5\%}(x)} x_k}{\sum_k x_k}.
$$
High values mean a small set of outside tokens explains most of the outside change (hotspots).


In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def topk_mass_row(x: np.ndarray, top_frac: float = 0.05, eps: float = 1e-12) -> float:
    """x: 1D nonnegative array. Return mass fraction in top top_frac entries."""
    s = float(x.sum())
    if s <= eps:
        return np.nan
    k = max(1, int(np.ceil(top_frac * x.size)))
    top = np.partition(x, -k)[-k:]  # O(n), faster than full sort
    return float(top.sum() / s)

@torch.no_grad()
def top5_mass_out_gpu(noisy_col: str, top_frac: float = 0.05, chunk_rows: int = 2048) -> np.ndarray:
    """
    Compute per-image outside concentration: top_frac mass of outside Δ.
    Uses GPU to compute Δ, CPU to aggregate per row.
    Returns (N,) float32.
    """
    z_noisy_np = np.asarray(df[noisy_col].to_list(), dtype=np.int32)  # (N, HW)
    N, HW = z_clean_np.shape
    out = np.empty((N,), dtype=np.float32)

    for s in tqdm(range(0, N, chunk_rows), desc=f"top{int(top_frac*100)}% mass (outside): {noisy_col}"):
        e = min(N, s + chunk_rows)

        zc = torch.from_numpy(z_clean_np[s:e]).to(device=device, dtype=torch.long)   # (B, HW)
        zn = torch.from_numpy(z_noisy_np[s:e]).to(device=device, dtype=torch.long)  # (B, HW)

        Ec = codebook_unit_t[zc]  # (B, HW, D)
        En = codebook_unit_t[zn]
        delta = ((1.0 - (Ec * En).sum(dim=-1).clamp(-1.0, 1.0)) * 0.5).float()      # (B, HW)
        delta_np = delta.cpu().numpy()

        # per-row concentration on outside tokens (outside mask differs per row)
        for b in range(e - s):
            outside_vals = delta_np[b][~inside_mask[s + b]]
            out[s + b] = topk_mass_row(outside_vals, top_frac=top_frac)

    return out

# Compute for all three noise levels
top5_out_low  = top5_mass_out_gpu("indices_low",  top_frac=0.05, chunk_rows=2048)
top5_out_mid  = top5_mass_out_gpu("indices_mid",  top_frac=0.05, chunk_rows=2048)
top5_out_high = top5_mass_out_gpu("indices_high", top_frac=0.05, chunk_rows=2048)

df_top5 = pd.DataFrame({
    "image_id": df["image_id"],
    "outside_low":  outside_low,
    "outside_mid":  outside_mid,
    "outside_high": outside_high,
    "ratio_low":    ratio_low,
    "ratio_mid":    ratio_mid,
    "ratio_high":   ratio_high,
    "top5_mass_out_low":  top5_out_low,
    "top5_mass_out_mid":  top5_out_mid,
    "top5_mass_out_high": top5_out_high,
})

# Quick summaries
display(df_top5[["top5_mass_out_low","top5_mass_out_mid","top5_mass_out_high"]].describe())
display(df_top5[["ratio_low","ratio_mid","ratio_high","outside_low","outside_mid","outside_high"]].describe())

# Correlations per sigma (outside vs concentration, ratio vs concentration)
corr_rows = []
for lvl in ["low","mid","high"]:
    corr_rows.append({
        "sigma": lvl,
        "corr(outside, top5_mass_out)": float(df_top5[[f"outside_{lvl}", f"top5_mass_out_{lvl}"]].corr().iloc[0,1]),
        "corr(ratio, top5_mass_out)":   float(df_top5[[f"ratio_{lvl}",   f"top5_mass_out_{lvl}"]].corr().iloc[0,1]),
    })
display(pd.DataFrame(corr_rows))


In [ ]:
plt.figure(figsize=(7,4))

data = [
    df_top5["top5_mass_out_low"].to_numpy(),
    df_top5["top5_mass_out_mid"].to_numpy(),
    df_top5["top5_mass_out_high"].to_numpy(),
]

plt.boxplot(data, labels=[f"σ={sigma_low}", f"σ={sigma_mid}", f"σ={sigma_high}"], showfliers=False)
plt.ylabel("top5_mass_out (outside concentration)")
plt.title("Outside concentration decreases with noise")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
def binned_mean(x, y, bins=40, min_count=200):
    x = np.asarray(x); y = np.asarray(y)
    edges = np.quantile(x, np.linspace(0, 1, bins+1))
    edges = np.unique(edges)
    xb, yb = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (x >= a) & (x <= b)
        if m.sum() < min_count:
            continue
        xb.append(float(x[m].mean()))
        yb.append(float(y[m].mean()))
    return np.array(xb), np.array(yb)

plt.figure(figsize=(7,4))

for lvl, sig in [("low", sigma_low), ("mid", sigma_mid), ("high", sigma_high)]:
    x = df_top5[f"outside_{lvl}"].to_numpy()
    y = df_top5[f"top5_mass_out_{lvl}"].to_numpy()
    xb, yb = binned_mean(x, y, bins=50, min_count=200)
    plt.plot(xb, yb, marker="o", markersize=3, label=f"σ={sig}")

plt.xlabel("outside_change (mean cosine distance outside patch)")
plt.ylabel("mean(top5_mass_out) within bins")
plt.title("Larger outside leakage tends to be more diffuse")
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


As outside leakage gets larger, it becomes more diffuse.

1. All curves trend downward: when outside_change increases, top5_mass_out decreases. Translation: bigger leakage is not “a few tokens going crazy”; it’s lots of outside tokens moving a bit.

2. Low noise has more hotspotty failures when leakage is tiny.
At σ=0.1 (blue), at very small outside_change you get very high top5_mass_out (0.7+). That’s “almost nothing changes outside, except a few tokens that change a lot.” Rare, spiky events.

3. At higher noise, outside leakage is already fairly diffuse.
For σ=0.5 (green), even at moderate outside_change you sit around 0.41–0.43 and then drop toward ~0.35 at the largest outside_change. So high σ leakage looks like global drift, not isolated hotspots.


Hypo: Across noise levels, larger outside leakage is associated with lower outside concentration: the outside response transitions from rare, spiky hotspots at low leakage to diffuse global drift as leakage increases, especially at higher σ.”

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
import torch

def draw_patch_bbox_px(ax, patch_bbox_px, linewidth=2):
    x0, y0, x1, y1 = map(int, patch_bbox_px)
    ax.add_patch(patches.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, linewidth=linewidth, edgecolor="black"))

@torch.no_grad()
def token_dist_grid_from_row(row, level: str, device=device):
    """HxW cosine embedding distance Δ in [0,1] for one row and noise level."""
    H, W = row["token_grid_hw"]
    zc = np.array(row["indices_clean"], dtype=np.int32).reshape(H, W)
    zn = np.array(row[f"indices_{level}"], dtype=np.int32).reshape(H, W)

    zc_t = torch.from_numpy(zc).to(device=device, dtype=torch.long)
    zn_t = torch.from_numpy(zn).to(device=device, dtype=torch.long)

    Ec = codebook_unit_t[zc_t]  # (H,W,D)
    En = codebook_unit_t[zn_t]
    cos_sim = (Ec * En).sum(dim=-1).clamp(-1.0, 1.0)
    delta = (1.0 - cos_sim) * 0.5  # (H,W) in [0,1]
    return delta.detach().cpu().numpy()

def show_full_3x4_for_idx(row_idx: int, overlay=True, vlim=(0,1)):
    """
    3x4 grid:
      Row 1: original + noisy inputs (low/mid/high)
      Row 2: recon clean + recon noisy (low/mid/high)
      Row 3: token embedding-distance heatmaps Δ (clean vs noisy) for low/mid/high (and blank under original)
    """
    row = df.loc[row_idx].to_dict()
    image_id = row["image_id"]
    files = find_full_set(image_id)

    print("\nimage_id:", image_id)
    print("found:", {k: files[k].name for k in sorted(files.keys())})

    needed = list(range(8))
    assert all(k in files for k in needed), "Not all 0..7 files found."

    titles_top = ["original", "input low", "input mid", "input high"]
    titles_mid = ["recon clean", "recon low", "recon mid", "recon high"]
    titles_bot = ["Δ heatmap", "Δ (low)", "Δ (mid)", "Δ (high)"]

    top_paths = [files[0], files[2], files[4], files[6]]
    mid_paths = [files[1], files[3], files[5], files[7]]

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))

    # Row 1: original + inputs
    for c, p in enumerate(top_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[0, c]
        ax.imshow(img)
        ax.set_title(titles_top[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_patch_bbox_px(ax, row["patch_bbox_px"])

    # Row 2: recons
    for c, p in enumerate(mid_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[1, c]
        ax.imshow(img)
        ax.set_title(titles_mid[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_patch_bbox_px(ax, row["patch_bbox_px"])

    # Row 3: Δ heatmaps (blank under "original"/"recon clean" column)
    # Col 0: blank (no noisy comparison)
    axes[2, 0].axis("off")
    axes[2, 0].set_title(titles_bot[0], fontsize=10)

    # Cols 1..3: low/mid/high
    levels = ["low", "mid", "high"]
    ims = []
    for c, lvl in enumerate(levels, start=1):
        delta = token_dist_grid_from_row(row, lvl)
        ax = axes[2, c]
        im = ax.imshow(delta, vmin=vlim[0], vmax=vlim[1])
        ax.set_title(titles_bot[c], fontsize=10)
        ax.axis("off")
        # token bbox overlay in token coords
        j0, i0, j1, i1 = row["patch_bbox_tok"]
        ax.add_patch(patches.Rectangle((j0-0.5, i0-0.5), (j1-j0), (i1-i0),
                                       fill=False, linewidth=2, edgecolor="black"))
        ims.append(im)

    # One shared colorbar for the Δ row
    cbar = fig.colorbar(ims[-1], ax=axes[2, :].ravel().tolist(), fraction=0.02, pad=0.01)
    cbar.set_label("cosine embedding distance Δ ∈ [0,1]")

    # Use constrained layout to avoid tight_layout warnings with colorbars
    fig.set_constrained_layout(True)
    plt.show()

print("LOW leakage sample recon+heatmap grid:")
show_full_3x4_for_idx(int(low_idxs[0]), overlay=True, vlim=(0,1))

print("MED leakage sample recon+heatmap grid:")
show_full_3x4_for_idx(int(med_idx), overlay=True, vlim=(0,1))

print("RANDOM leakage sample recon+heatmap grid:")
show_full_3x4_for_idx(int(rand_idxs[0]), overlay=True, vlim=(0,1))

print("HIGH leakage sample recon+heatmap grid:")
show_full_3x4_for_idx(int(high_idxs[0]), overlay=True, vlim=(0,1))

In [ ]:
x